# Assignment 1: Prompt-Only Pricing Agent (Baseline)

## Objective
Build a simple LLM price recommender using **prompts only** - no tools, no data lookup, just pure LLM reasoning!

## Requirements
**User Input:**
- Cost Price
- Current Price
- Target Margin (%)
- Competitor Price
- Price Elasticity (Low/Medium/High)

**Agent Action:**
- Computes recommended price using verbal reasoning
- No tools, no data lookup
- Clear explanation of reasoning

## ️Setup & Dependencies

First, let's install the required packages and set up our environment.

In [1]:
# Install required packages
!pip install -q langchain langchain-groq

In [2]:
# Import required libraries
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage
import os
import getpass

In [3]:
# Set up your Groq API key
print("🔑 Please enter your Groq API key:")
print("(You can get one free at: https://console.groq.com/)")
groq_api_key = getpass.getpass("Groq API Key: ")
os.environ["GROQ_API_KEY"] = groq_api_key
print("✅ API key set successfully!")

🔑 Please enter your Groq API key:
(You can get one free at: https://console.groq.com/)
✅ API key set successfully!


## Initialize the Language Model

Let's set up our LLM for pricing recommendations.

In [4]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)

## Prompt Engineering for Pricing

Now let's create effective prompts for our pricing agent. We'll start with a basic approach and then improve it.

### Basic Prompt 

**Task:** Create a basic pricing prompt template. Fill in the template below.

In [27]:
# Basic pricing prompt template
basic_pricing_prompt = PromptTemplate(
    input_variables=["cost_price", "current_price", "target_margin", "competitor_price", "price_elasticity"],
    template="""You are a pricing expert. Based on the following product pricing data, recommend an optimal selling price.

Cost Price: ${cost_price}
Current Price: ${current_price}
Target Margin: {target_margin}%
Competitor Price: ${competitor_price}
Price Elasticity: {price_elasticity}

Provide a recommended price and a brief explanation of your reasoning."""
)

### Test the Basic Prompt

Let's test our basic prompt with the example from the assignment.

In [28]:
# Test with the assignment example
test_input = {
    "cost_price": 400,
    "current_price": 599,
    "target_margin": 25,
    "competitor_price": 579,
    "price_elasticity": "Medium"
}

basic_chain = basic_pricing_prompt | llm
basic_response = basic_chain.invoke(test_input)

print("📊 Basic Pricing Recommendation:")
print(basic_response.content)

📊 Basic Pricing Recommendation:
Based on the provided data, I recommend an optimal selling price of $549.

My reasoning is as follows:

1. The target margin is 25%, which means the desired selling price should be the cost price plus 25% of the cost price. Calculating this, we get: $400 + (25% of $400) = $400 + $100 = $500.
2. However, considering the competitor's price is $579, we may not want to price our product significantly lower, as it may be perceived as lower quality. A medium price elasticity suggests that customers are somewhat sensitive to price changes, but not extremely so.
3. The current price is $599, which is higher than the competitor's price. Reducing the price to $549 would still maintain a premium positioning while being more competitive.
4. The recommended price of $549 is higher than the calculated target price of $500, but it balances the need to achieve the target margin with the need to be competitive in the market.

By pricing the product at $549, we can mainta

### Improved Prompt with Chain-of-Thought

**Task:** Create an improved prompt that uses chain-of-thought reasoning for better explanations.

In [20]:

improved_pricing_prompt = PromptTemplate(
    input_variables=["cost_price", "current_price", "target_margin", "competitor_price", "price_elasticity"],
    template="""You are a pricing expert. Based on the following product pricing data, recommend an optimal selling price.

Cost Price: ${cost_price}
Current Price: ${current_price}
Target Margin: {target_margin}%
Competitor Price: ${competitor_price}
Price Elasticity: {price_elasticity}

Think through this step-by-step:

Step 1 - Calculate minimum price based on target margin:
Calculate cost_price * (1 + target_margin/100) to find the floor price.

Step 2 - Analyze competitive positioning:
Compare the minimum price with the competitor price. Are we above or below?

Step 3 - Consider price elasticity impact:
If elasticity is High, price closer to or below competitor. If Low, we have room to price higher.

Step 4 - Recommend optimal price:
Based on all the above analysis, provide your final recommended price.

Show your work for each step with actual numbers."""
)


In [21]:
improved_chain = improved_pricing_prompt | llm
improved_response = improved_chain.invoke(test_input)

print("📈 Improved Pricing Analysis:")
print(improved_response.content)

📈 Improved Pricing Analysis:
To determine the optimal selling price, let's follow the steps outlined:

### Step 1 - Calculate minimum price based on target margin:
- Cost Price: $400
- Target Margin: 25%
- Minimum Price = Cost Price * (1 + Target Margin/100)
- Minimum Price = $400 * (1 + 25/100)
- Minimum Price = $400 * 1.25
- Minimum Price = $500

### Step 2 - Analyze competitive positioning:
- Minimum Price: $500
- Competitor Price: $579
- Current Price: $599
- We are above both the minimum price and the competitor's price.

### Step 3 - Consider price elasticity impact:
- Price Elasticity: Medium
- Since the elasticity is medium, we need to balance between pricing closer to the competitor to remain competitive and maintaining a higher price to maximize revenue. Given that our current price is already higher than the competitor's, and considering medium elasticity, we should aim to reduce the price to be more competitive but not necessarily match or go below the competitor's price.



## Class-Based Pricing Agent

Now let's create a reusable class-based pricing agent, similar to the PhysicsTeacherAgent structure.

In [29]:
class PricingAgent:
    def __init__(self):
        # TODO: Initialize the LLM
        self.llm = llm

        # TODO: Create the system message for pricing agent persona
        self.system_message = SystemMessage(
            content="You are a highly skilled pricing agent with expertise in market analysis, competitive pricing, and profit optimization. Your goal is to provide actionable pricing recommendations based on input data."
            )
        
        # TODO: Create the pricing prompt template
        self.pricing_template = """You are a pricing expert. Based on the following product pricing data, recommend an optimal selling price.   
        Cost Price: ${cost_price}
        Current Price: ${current_price}
        Target Margin: {target_margin}%
        Competitor Price: ${competitor_price}
        Price Elasticity: {price_elasticity}"""
        pass
        
    def get_price_recommendation(self, cost_price, current_price, target_margin, competitor_price, price_elasticity):
        """Get price recommendation with detailed analysis"""
        # TODO: Implement the price recommendation logic
        messages = [
            self.system_message,
            HumanMessage(content=self.pricing_template.format(
                cost_price=cost_price,
                current_price=current_price,
                target_margin=target_margin,
                competitor_price=competitor_price,
                price_elasticity=price_elasticity
            ))
        ]
        
        response = self.llm.invoke(messages)
        return response.content
    
    def quick_price_check(self, cost_price, target_margin, competitor_price):
        """Quick price check with minimal inputs"""
        quick_prompt = f"""Quick pricing check:
        Cost: ${cost_price}, Target Margin: {target_margin}%, Competitor: ${competitor_price}
        
        Provide a quick price recommendation with brief reasoning."""
        
        response = self.llm.invoke([HumanMessage(content=quick_prompt)])
        return response.content

print("🏗️ Initializing Pricing Agent...")
pricing_agent = PricingAgent()
print("✅ Pricing Agent Ready!")

🏗️ Initializing Pricing Agent...
✅ Pricing Agent Ready!


## Testing Our Pricing Agent

Let's test our pricing agent with various scenarios.

In [30]:
# Test with the assignment example
print("Testing Assignment Example:")
print("=" * 50)

result = pricing_agent.get_price_recommendation(
    cost_price=400,
    current_price=599,
    target_margin=25,
    competitor_price=579,
    price_elasticity="Medium"
)

print(result)

Testing Assignment Example:
To determine the optimal selling price, let's analyze the given data:

1. **Cost Price**: $400
2. **Current Price**: $599
3. **Target Margin**: 25%
4. **Competitor Price**: $579
5. **Price Elasticity**: Medium

**Step 1: Calculate the Desired Selling Price based on Target Margin**

To achieve a 25% target margin, we need to calculate the selling price as follows:

Desired Selling Price = Cost Price / (1 - Target Margin)
= $400 / (1 - 0.25)
= $400 / 0.75
= $533.33

**Step 2: Consider the Competitor Price**

The competitor's price is $579, which is higher than our desired selling price of $533.33. Since the price elasticity is medium, we can assume that a moderate price change will have a moderate impact on demand.

**Step 3: Balance Competitor Price and Target Margin**

To balance the competitor price and our target margin, we can consider a price that is closer to the competitor's price while still maintaining a reasonable margin. Let's calculate the selling

In [ ]:
# Test Case 1: High elasticity scenario
print("\nTest Case 1: High Elasticity Scenario")
print("=" * 50)
result1 = pricing_agent.get_price_recommendation(
    cost_price=100,
    current_price=200,
    target_margin=30,
    competitor_price=180,
    price_elasticity="High"
)
print(result1)

# Test Case 2: Low elasticity scenario  
print("\nTest Case 2: Low Elasticity Scenario")
print("=" * 50)
result2 = pricing_agent.get_price_recommendation(
    cost_price=50,
    current_price=100,
    target_margin=40,
    competitor_price=120,
    price_elasticity="Low"
)
print(result2)

# Test Case 3: Custom  
print("\nTest Case 3: Custom")
print("=" * 50)
result3 = pricing_agent.get_price_recommendation(
    cost_price=70,
    current_price=120,
    target_margin=20,
    competitor_price=60,
    price_elasticity="Low"
)
print(result3)


Test Case 1: High Elasticity Scenario
To determine the optimal selling price, let's analyze the given data:

1. **Cost Price**: $100
2. **Current Price**: $200
3. **Target Margin**: 30%
4. **Competitor Price**: $180
5. **Price Elasticity**: High

**Step 1: Calculate the Desired Selling Price based on Target Margin**

To achieve a 30% margin, we need to calculate the selling price as follows:

Desired Selling Price = Cost Price / (1 - Target Margin)
= $100 / (1 - 0.30)
= $100 / 0.70
= $142.86

However, this price is lower than the competitor's price and the current price. We need to consider the competitor's price and price elasticity.

**Step 2: Consider Competitor Price and Price Elasticity**

Given the high price elasticity, it means that customers are highly sensitive to price changes. A small increase in price may lead to a significant decrease in demand.

Considering the competitor's price of $180, we may not want to price our product significantly higher than this. However, our 

In [33]:
# Test the quick price check function
print("⚡ Quick Price Check Test:")
print("=" * 30)

quick_result = pricing_agent.quick_price_check(
    cost_price=250,
    target_margin=20,
    competitor_price=350
)
print(quick_result)

⚡ Quick Price Check Test:
Based on the given information, I would recommend a price of $300. 

This price is higher than the cost plus target margin ($250 + 20% of $250 = $300), which would be $300, exactly meeting the target margin. It's also lower than the competitor's price of $350, making it competitive while maintaining the desired profit margin.


## Experiment with Different Prompting Strategies

Try different prompting approaches and compare the results.

In [34]:
few_shot_template = PromptTemplate(
    input_variables=["cost_price", "current_price", "target_margin", "competitor_price", "price_elasticity"],
    template="""You are a pricing expert. Here are some examples of good pricing decisions:

Example 1:
Cost: $200, Current: $400, Target Margin: 30%, Competitor: $380, Elasticity: Medium
Recommendation: $375 (maintains margin above 30%, competitive with market, good for medium elasticity)

Example 2:  
Cost: $100, Current: $180, Target Margin: 25%, Competitor: $200, Elasticity: Low
Recommendation: $190 (exceeds margin target, leverages low elasticity for higher profit)

Now analyze this case:
Cost: ${cost_price}, Current: ${current_price}, Target Margin: {target_margin}%, Competitor: ${competitor_price}, Elasticity: {price_elasticity}

Recommendation:"""
)

# Test few-shot approach
few_shot_chain = few_shot_template | llm
few_shot_result = few_shot_chain.invoke(test_input)

print("🎯 Few-Shot Prompting Result:")
print(few_shot_result.content)

🎯 Few-Shot Prompting Result:
To determine the recommended price, let's analyze the given information:

- Cost: $400
- Current Price: $599
- Target Margin: 25%
- Competitor Price: $579
- Elasticity: Medium

First, calculate the minimum price that meets the target margin:
Target Margin = 25%
Target Price = Cost / (1 - Target Margin) = $400 / (1 - 0.25) = $400 / 0.75 = $533.33

Since the target price ($533.33) is lower than both the current price ($599) and the competitor's price ($579), and considering the medium elasticity, we should aim for a price that is competitive with the market while maintaining the target margin.

Given that the competitor's price is $579, which is higher than the calculated target price ($533.33), and considering medium elasticity, we can recommend a price that is slightly below the competitor's price to maintain competitiveness while ensuring the target margin is met or exceeded.

Recommendation: $575

This price is competitive with the market (slightly below 

## Compare Different Approaches

Let's compare the different prompting strategies we've implemented.

In [36]:
# TODO: Create a comparison of all approaches
def compare_pricing_approaches(cost_price, current_price, target_margin, competitor_price, price_elasticity):
    """Compare different prompting approaches for the same input"""
    
    print(f"📊 PRICING COMPARISON FOR:")
    print(f"Cost: ${cost_price}, Current: ${current_price}, Target Margin: {target_margin}%")
    print(f"Competitor: ${competitor_price}, Elasticity: {price_elasticity}")
    print("="*60)
    
    # Basic approach
    basic_result = basic_chain.invoke({
        "cost_price": cost_price,
        "current_price": current_price,
        "target_margin": target_margin,
        "competitor_price": competitor_price,
        "price_elasticity": price_elasticity
    })
    
    # Improved approach
    improved_result = improved_chain.invoke({
        "cost_price": cost_price,
        "current_price": current_price,
        "target_margin": target_margin,
        "competitor_price": competitor_price,
        "price_elasticity": price_elasticity
    })
    
    # Class-based approach
    class_result = pricing_agent.get_price_recommendation(
        cost_price, current_price, target_margin, competitor_price, price_elasticity
    )
    
    print("🔴 BASIC APPROACH:")
    print(basic_result.content[:500] + "...\n")
    
    print("🟡 IMPROVED APPROACH:")
    print(improved_result.content[:500] + "...\n")
    
    print("🟢 CLASS-BASED APPROACH:")
    print(class_result[:500] + "...")

# Run comparison
compare_pricing_approaches(400, 599, 25, 579, "Medium")

📊 PRICING COMPARISON FOR:
Cost: $400, Current: $599, Target Margin: 25%
Competitor: $579, Elasticity: Medium
🔴 BASIC APPROACH:
Based on the provided data, I recommend an optimal selling price of $549.

My reasoning is as follows:

1. The target margin is 25%, which means the desired selling price should be the cost price plus 25% of the cost price. Calculating this, we get: $400 + (25% of $400) = $400 + $100 = $500.
2. However, considering the competitor's price is $579, we may not want to price our product significantly lower, as it may be perceived as lower quality. A medium price elasticity suggests that customers ar...

🟡 IMPROVED APPROACH:
To determine the optimal selling price, let's follow the steps outlined:

### Step 1: Calculate Minimum Price Based on Target Margin

Given:
- Cost Price = $400
- Target Margin = 25%

Minimum Price = Cost Price * (1 + Target Margin/100)
Minimum Price = $400 * (1 + 25/100)
Minimum Price = $400 * 1.25
Minimum Price = $500

So, the minimum price ba